# LAB3 — Bài 5 (trang 29): Wisconsin Breast Cancer

**Sinh viên:** Trần Chí Nguyên — **MSSV:** 6451071052.

> Mỗi câu lập trình in họ tên và MSSV theo template. Chạy lần lượt từ trên xuống.

In [1]:
%matplotlib inline
HO_TEN = 'Trần Chí Nguyên'
MSSV = '6451071052'
def in_danh_tinh(): print(f'{HO_TEN} - {MSSV}')


## Câu 5 trang 29. Gom cụm Wisconsin Diagnostic Breast Cancer

Đọc bộ dữ liệu WDBC từ Google Drive, gom cụm thành 2 nhóm bằng k-Means, rồi so sánh với chẩn đoán B/M bằng các độ đo yêu cầu.

In [2]:
in_danh_tinh()
import io, urllib.request, zipfile
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import confusion_matrix, classification_report, adjusted_rand_score, normalized_mutual_info_score, silhouette_score
from scipy.optimize import linear_sum_assignment
cols=['id','diagnosis']+[f'feature_{i}' for i in range(1,31)]
DATA_URL='https://drive.google.com/uc?export=download&id=13qn-5-ccmFiPv4oPxcLxFol1KvRzk-L5'
with urllib.request.urlopen(DATA_URL,timeout=60) as response:
    archive=zipfile.ZipFile(io.BytesIO(response.read()))
    df=pd.read_csv(io.BytesIO(archive.read('wdbc.data')),header=None,names=cols)
print('Shape:',df.shape); display(df.head()); print(df['diagnosis'].value_counts())
y=(df['diagnosis']=='M').astype(int).to_numpy()
X=df.drop(columns=['id','diagnosis']).to_numpy(dtype=float)
X_scaled=StandardScaler().fit_transform(X)
km=KMeans(n_clusters=2,random_state=42,n_init=20)
cluster=km.fit_predict(X_scaled)
print('Silhouette:',silhouette_score(X_scaled,cluster)); print('ARI:',adjusted_rand_score(y,cluster)); print('NMI:',normalized_mutual_info_score(y,cluster))
raw=confusion_matrix(y,cluster,labels=[0,1]); rows,cols_idx=linear_sum_assignment(-raw)
mapping={c:r for r,c in zip(rows,cols_idx)}
aligned=np.array([mapping[c] for c in cluster])
cm=confusion_matrix(y,aligned,labels=[0,1])
print('Ánh xạ cụm -> chẩn đoán (0=B, 1=M):',mapping)
print(classification_report(y,aligned,target_names=['Benign','Malignant'],zero_division=0,digits=4))
sns.heatmap(cm,annot=True,fmt='d',cmap='Blues',xticklabels=['Benign','Malignant'],yticklabels=['Benign','Malignant'])
plt.xlabel('Cụm đã ánh xạ'); plt.ylabel('Chẩn đoán thật'); plt.title('Wisconsin Breast Cancer — Confusion matrix'); plt.show()

Trần Chí Nguyên - 6451071052


Shape: (569, 32)


,id,diagnosis,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,...,feature_21,feature_22,feature_23,feature_24,feature_25,feature_26,feature_27,feature_28,feature_29,feature_30
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


diagnosis
B    357
M    212
Name: count, dtype: int64


Silhouette: 0.3449740051034408
ARI: 0.6707206476880808
NMI: 0.554612218410556
Ánh xạ cụm -> chẩn đoán (0=B, 1=M): {np.int64(1): np.int64(0), np.int64(0): np.int64(1)}
              precision    recall  f1-score   support

      Benign     0.9026    0.9608    0.9308       357
   Malignant     0.9259    0.8255    0.8728       212

    accuracy                         0.9104       569
   macro avg     0.9143    0.8931    0.9018       569
weighted avg     0.9113    0.9104    0.9092       569



## Nhận xét

So sánh chỉ số tổng thể và theo lớp; k-Means tối ưu khoảng cách Euclidean nên chuẩn hóa đặc trưng là cần thiết. Nhãn cụm được hoán vị để khớp nhãn B/M tối đa trước khi tính confusion matrix.